####  BOKU WALDBRAND-DATEN LADEN & VORBEREITEN
####  GEOSPHERE HISTORICAL STATION DATA LADEN
####  DER GEO-TEMPORAL JOIN (Wetterdaten pro Brand extrahieren)


##### Die Bibliotheken installieren ( Geospatial-Bibliotheken für Python)

In [ ]:
# pip install geopandas 

#### Stationsdaten-v2 (1 d): Stationsdaten für Österreich in täglicher Auflösung mit Qualitätsinformationen
##### See [https://data.hub.geosphere.at/dataset/klima-v2-1d](https://data.hub.geosphere.at/dataset/klima-v2-1d)

In [4]:
import requests
import io
import pandas as pd
import numpy as np
import json
import time
from scipy.spatial import KDTree

In [ ]:
# API-Struktur 
# https://dataset.api.hub.geosphere.at/v1/station/historical/klima-v2-1d
url = "https://dataset.api.hub.geosphere.at"
version = "v1"
type_ = "station"
mode = "historical"
resource_id = "klima-v2-1d"
endpoint = f"{url}/{version}/{type_}/{mode}/{resource_id}"

# Zeitfenster: 14 Tage vor dem Brand (11.06.2014) bis zum Brandtag (25.06.2014)
# objid,waldbrandid,jahr,monat,tag,bundesland,gemeinde,brandart,brandflaeche,ursache1,feuerwehren,einsatzkraefte,quelle1,verifiziert,gemeinde_neu,brandflaeche_korr,schutzwald,longitude,latitude
# 6125,2014062501,2014,6,25,NOE,Untersiebenbrunn,2,100000,1,4,-9999,http://www.feuerwehr-untersiebenbrunn.at/,True,Untersiebenbrunn,0.0,1,16.75,48.274

output_file = "../data/final_waldbrand+geosphere_klima_dataset_teil_011-015.csv"  # "../data/final_waldbrand+geosphere_klima_dataset.csv"
metadata_input_file_path = "../data/response_dataset.api.hub.geosphere.at_v1_station_historical_klima-v2-1d_metadata.json"
input_file_path = "../data/teil_011-015.csv"  # "../data/boku_waldbrand_02-11-2026(teil_011-015).csv" # "../data/boku_waldbrand_02-11-2026.csv"
timewindow_days_per_case = 14

#  METADATEN LADEN & NUR GÜLTIGE "COMBINED"-STATIONEN FILTERN
# hochgeladene Metadaten-Datei response_dataset.api.hub.geosphere.at_v1_station_historical_klima-v2-1d_metadata.json
with open(metadata_input_file_path, "r", encoding="utf-8") as f:
    meta_data = json.load(f)

stations_list = []
for s in meta_data.get("stations", []):
    # 'COMBINED'-Stationen abdecken lange historische Zeiträume 
    if s.get("type") == "COMBINED":
        stations_list.append({
            "station_id": s["id"],
            "name": s["name"],
            "lat": float(s["lat"]),
            "lon": float(s["lon"]),
            "valid_from": pd.to_datetime(s["valid_from"]).tz_localize(None),
            "valid_to": pd.to_datetime(s["valid_to"]).tz_localize(None)
        })

stations_df = pd.DataFrame(stations_list)
print(f">>> {len(stations_df)} kombinierte Hauptstationen geladen")

# WALDBRAND-DATEN LADEN & VORBEREITEN
waldbrand_df = pd.read_csv(input_file_path)
waldbrand_df = waldbrand_df.dropna(subset=['jahr', 'monat', 'tag', 'longitude', 'latitude'])

date_bridge = pd.DataFrame({'year': waldbrand_df['jahr'], 'month': waldbrand_df['monat'], 'day': waldbrand_df['tag']})
waldbrand_df['date'] = pd.to_datetime(date_bridge, errors='coerce')
waldbrand_df = waldbrand_df.dropna(subset=['date'])

# API-ABFRAGE MIT EXAKTEN PARAMETERN & ZEIT-VALIDIERUNG
# "https://geosphere.at"

# 12 Parameter 
target_parameters = [
    "tl_ii", "tlmax", "tl_mittel",
    "vvbft_ii", "vv_mittel",
    "rfb_ii", "rfb_mittel", "rf_ii", "rf_mittel",
    "rr_iii", "sh", "p_mittel"
]

extracted_results = []
print(f"Start Download für {len(waldbrand_df)} Waldbrand-Ereignisse...\n")

for idx, fire in waldbrand_df.iterrows():
    fire_date = fire['date']
    fire_id = fire['waldbrandid']
    
    # Zeitfenster: X Tage vor dem Brand bis zum Brandtag selbst
    start_date = fire_date - pd.Timedelta(days=timewindow_days_per_case)
    
    # Filter: Nur Stationen berücksichtigen, die in diesem Zeitraum auch aktiv waren!
    valid_stations = stations_df[
        (stations_df['valid_from'] <= start_date) & 
        (stations_df['valid_to'] >= fire_date)
    ].reset_index(drop=True)
    
    if valid_stations.empty:
        print(f"Waldbrand ID {fire_id}: Keine aktive Station für Datum {fire_date.strftime('%Y-%m-%d')} gefunden")
        continue

    # Räumliche Suche nach der nächstgelegenen gültigen Station
    tree = KDTree(valid_stations[['lon', 'lat']].values)
    _, nearest_idx = tree.query([float(fire['longitude']), float(fire['latitude'])])
    best_station = valid_stations.iloc[nearest_idx]
    station_id = best_station['station_id']
    
    params = {
        "parameters": target_parameters,  # Requests wandelt Listen automatisch in ?parameters=a&parameters=b um
        "start": start_date.strftime('%Y-%m-%dT00:00:00'),
        "end": fire_date.strftime('%Y-%m-%dT00:00:00'),
        "station_ids": str(station_id),
        "output_format": "csv"
    }
    
    try:
        response = requests.get(endpoint, params=params)
        if response.status_code == 200:
            raw_text = response.text.strip()
            
            # Falls HTML zurückkommt
            if raw_text.startswith("<"):
                print(f">>> ID {fire_id}: HTML-Umleitung erhalten (URL: {response.url})")
                continue
                
            # CSV einlesen
            weather_df = pd.read_csv(io.StringIO(raw_text))
            
            # Prüfen, ob nur der Header da ist, aber 0 Datenzeilen
            if weather_df.empty:
                print(f">>> ID {fire_id} (Station {station_id}): CSV hat 0 Datenzeilen (Keine Messwerte in diesem Zeitraum).")
                continue
                
            # Spaltennamen sicherheitshalber kleinschreiben & säubern
            weather_df.columns = [c.strip().lower() for c in weather_df.columns]
            
            # Letzte Zeile = Brandtag | Vorherige Zeilen = 14 Vortage
            fire_day_data = weather_df.iloc[-1].to_dict()
            prev_14d_data = weather_df.iloc[:-1]
            
            row_result = {
                "boku_waldbrandid": fire_id,
                "matched_station_id": station_id,
                "matched_station_name": best_station['name'],
                "tl_ii_day": fire_day_data.get("tl_ii"),
                "tlmax_day": fire_day_data.get("tlmax"),
                "tl_mittel_day": fire_day_data.get("tl_mittel"),
                "vvbft_ii_day": fire_day_data.get("vvbft_ii"),
                "vv_mittel_day": fire_day_data.get("vv_mittel"),
                "rfb_ii_day": fire_day_data.get("rfb_ii"),
                "rfb_mittel_day": fire_day_data.get("rfb_mittel"),
                "rf_ii_day": fire_day_data.get("rf_ii"),
                "rf_mittel_day": fire_day_data.get("rf_mittel"),
                "rr_iii_day": fire_day_data.get("rr_iii"),
                "sh_day": fire_day_data.get("sh"),
                "p_mittel_day": fire_day_data.get("p_mittel")
            }
            
            # Feature Engineering für die 14 Vortage
            if not prev_14d_data.empty and "rr_iii" in prev_14d_data.columns:
                rr_clean = pd.to_numeric(prev_14d_data["rr_iii"], errors='coerce').replace(-1, 0)
                row_result["rr_iii_sum_14d"] = rr_clean.sum()
                row_result["tlmax_max_14d"] = pd.to_numeric(prev_14d_data["tlmax"], errors='coerce').max()
                row_result["rf_ii_min_14d"] = pd.to_numeric(prev_14d_data["rf_ii"], errors='coerce').min()
                row_result["vv_mittel_mean_14d"] = pd.to_numeric(prev_14d_data["vv_mittel"], errors='coerce').mean()
            
            extracted_results.append(row_result)
            print(f"Waldbrand ID {fire_id} ({fire_date.strftime('%Y-%m-%d')}) -> Station {station_id} ({best_station['name']}) OK ({len(weather_df)} Tage geladen)")
            
        elif response.status_code == 429:
            print(">>> Rate Limit erreicht (240 Requests/Stunde)! Warte 60s")
            time.sleep(60)
        else:
            print(f">>> Waldbrand ID {fire_id}: API Status {response.status_code} | Text: {response.text[:100]}")
            
    except Exception as e:
        print(f">>> Exception bei Waldbrand ID {fire_id}: {e}")
    
        '''if response.status_code == 200 and not response.text.strip().startswith("<"):
            weather_df = pd.read_csv(io.StringIO(response.text.strip()))
            
            if not weather_df.empty:
                # Letzte Zeile = Tag des Brandausbruchs
                fire_day_data = weather_df.iloc[-1].to_dict()
                # Alle Zeilen davor = Die 14 Vortage (Dürrephase)
                prev_14d_data = weather_df.iloc[:-1]
                
                # Metadaten & Brandtag-Werte speichern
                row_result = {
                    "boku_waldbrandid": fire_id,
                    "matched_station_id": station_id,
                    "matched_station_name": best_station['name'],
                    "tl_ii_day": fire_day_data.get("tl_ii"),
                    "tlmax_day": fire_day_data.get("tlmax"),
                    "tl_mittel_day": fire_day_data.get("tl_mittel"),
                    "vvbft_ii_day": fire_day_data.get("vvbft_ii"),
                    "vv_mittel_day": fire_day_data.get("vv_mittel"),
                    "rfb_ii_day": fire_day_data.get("rfb_ii"),
                    "rfb_mittel_day": fire_day_data.get("rfb_mittel"),
                    "rf_ii_day": fire_day_data.get("rf_ii"),
                    "rf_mittel_day": fire_day_data.get("rf_mittel"),
                    "rr_iii_day": fire_day_data.get("rr_iii"),
                    "sh_day": fire_day_data.get("sh"),
                    "p_mittel_day": fire_day_data.get("p_mittel")
                }
                
                # Feature Engineering für die 14 Vortage (Achtung: -1 bei rr_iii bedeutet 'kein Regen' -> für Summe auf 0 setzen!)
                if not prev_14d_data.empty:
                    rr_clean = prev_14d_data["rr_iii"].replace(-1, 0)
                    row_result["rr_iii_sum_14d"] = rr_clean.sum()
                    row_result["tlmax_max_14d"] = prev_14d_data["tlmax"].max()
                    row_result["rf_ii_min_14d"] = prev_14d_data["rf_ii"].min()
                    row_result["vv_mittel_mean_14d"] = prev_14d_data["vv_mittel"].mean()
                
                extracted_results.append(row_result)
                print(f"Waldbrand ID {fire_id} ({fire_date.strftime('%Y-%m-%d')}) -> Station {station_id} ({best_station['name']}) OK")
        else:
            print(f"! Waldbrand ID {fire_id}: API Status {response.status_code}")
            
    except Exception as e:
        print(f">>>! Fehler bei Waldbrand ID {fire_id}: {e}")
        
    time.sleep(0.05) # Schonendes Rate-Limiting'''
    time.sleep(0.35)

# DATENSÄTZE ZUSAMMENFÜHREN & SPEICHERN
results_df = pd.DataFrame(extracted_results)
if not results_df.empty:
    final_ml_df = waldbrand_df.merge(results_df, left_on="waldbrandid", right_on="boku_waldbrandid")
    final_ml_df.to_csv(output_file, index=False)
    print(f"\nFertig! {len(final_ml_df)} Datensätze gespeichert unter '{output_file}'")

    # Der HTTP-Statuscode 404 Not Found
    # Der HTTP-Statuscode 403 Forbidden 


>>> 216 kombinierte Hauptstationen geladen
Start Download für 26 Waldbrand-Ereignisse...

[+] ID 2015120702 (2015-12-07) -> Station 69 (Obervellach) OK (15 Tage geladen)
[+] ID 2011071101 (2011-07-11) -> Station 198 (Pörtschach) OK (15 Tage geladen)
[+] ID 2011033001 (2011-03-30) -> Station 178 (Hochfilzen) OK (15 Tage geladen)
[+] ID 2014122401 (2014-12-24) -> Station 138 (Prutz) OK (15 Tage geladen)
[+] ID 2013080101 (2013-08-01) -> Station 112 (Wiener Neustadt Flugplatz) OK (15 Tage geladen)
[+] ID 2012121101 (2012-12-11) -> Station 93 (St.Pölten Landhaus) OK (15 Tage geladen)
[+] ID 2013091201 (2013-09-12) -> Station 189 (Melk) OK (15 Tage geladen)
[+] ID 2013072916 (2013-07-29) -> Station 183 (Güssing) OK (15 Tage geladen)
[+] ID 2013072906 (2013-07-29) -> Station 217 (Kapfenberg Flugfeld) OK (15 Tage geladen)
[+] ID 2011030402 (2011-03-04) -> Station 187 (Friesach) OK (15 Tage geladen)
[+] ID 2013040701 (2013-04-07) -> Station 136 (Umhausen) OK (15 Tage geladen)
[+] ID 2015072502